# 🚀 PEFT / QLoRA Fine-Tuning — Complete Colab Notebook

> **GPU Required**: Go to Runtime → Change Runtime Type → T4 GPU (free tier)  
> **Model**: TinyLlama-1.1B (fits T4 easily) — swap to Mistral-7B on T4 Pro  
> **Task**: Medical Q&A instruction tuning (replaceable with any dataset)  
> **Framework**: Hugging Face PEFT + TRL + BitsAndBytes

---
## 📋 What We'll Do
1. Install all libraries
2. Load & prepare dataset
3. Load tokenizer
4. Load quantized base model (4-bit QLoRA)
5. Apply LoRA adapters
6. Train with SFTTrainer
7. Evaluate: Perplexity, ROUGE, BLEU, qualitative comparison
8. Save & optionally merge adapter

## Step 1: Check GPU & Install Libraries

In [1]:
# ── Check GPU ──────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f"\n✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Sun Mar 22 15:23:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              8W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── Install all required libraries ────────────────────────
# This may take 2-3 minutes
!pip install -q -U \
    transformers \
    datasets \
    peft \
    trl \
    bitsandbytes \
    accelerate \
    evaluate \
    rouge_score \
    nltk \
    sentencepiece \
    scipy

print("✅ All libraries installed!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 101.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.0 MB/s eta 0:00:00
✅ All libraries installed!


In [3]:
# ── Verify versions ────────────────────────────────────────
import transformers, peft, trl, bitsandbytes, datasets
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"trl          : {trl.__version__}")
print(f"bitsandbytes : {bitsandbytes.__version__}")
print(f"datasets     : {datasets.__version__}")

transformers : 5.3.0
peft         : 0.18.1
trl          : 0.29.1
bitsandbytes : 0.49.2
datasets     : 4.8.3


## Step 2: Import Libraries & Set Configuration

> 🔧 **All your hyperparameters are in one config cell below. Change here, not scattered throughout.**

In [4]:
import os
import torch
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
import evaluate
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("✅ All imports successful!")

✅ All imports successful!


In [5]:
# ══════════════════════════════════════════════════════════
#   🎛️  MASTER CONFIGURATION — Edit your hyperparameters here
# ══════════════════════════════════════════════════════════

# ── Model Config ──────────────────────────────────────────
# TinyLlama: perfect for free T4 (1.1B params, fits in 4GB)
# Swap to "mistralai/Mistral-7B-v0.1" on paid T4 or A10
BASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = "./qlora-finetuned-model"
ADAPTER_DIR = "./qlora-adapter-only"

# ── Dataset Config ────────────────────────────────────────
# We'll use a small medical Q&A dataset
# Replace with your own dataset using the same format
DATASET_NAME = "medalpaca/medical_meadow_wikidoc_patient_information"
MAX_TRAIN_SAMPLES = 500   # ← Limit for fast demo; use None for full dataset
MAX_VAL_SAMPLES   = 100
TEST_SIZE         = 0.1   # 10% validation split

# ── LoRA Hyperparameters ──────────────────────────────────
LORA_R          = 16      # Rank: 8=less capacity/memory, 32=more capacity/memory
LORA_ALPHA      = 32      # Scale = alpha/r. Convention: alpha = 2 * r
LORA_DROPOUT    = 0.05    # Dropout for LoRA layers (regularization)
# Which layers to apply LoRA to
# TinyLlama layers: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj
TARGET_MODULES  = ["q_proj", "k_proj", "v_proj", "o_proj"]

# ── Quantization (QLoRA) ─────────────────────────────────
LOAD_IN_4BIT         = True         # Enable QLoRA (4-bit quantization)
BNB_COMPUTE_DTYPE    = torch.float16  # Computation dtype (float16 for T4)
BNB_QUANT_TYPE       = "nf4"        # NormalFloat4 — best for LLM weights
BNB_DOUBLE_QUANT     = True         # Double quantization saves ~0.4 bits extra

# ── Training Hyperparameters ─────────────────────────────
NUM_EPOCHS           = 2            # Training epochs (1-3 for small datasets)
BATCH_SIZE           = 2            # Per-GPU batch size (reduce if OOM)
GRAD_ACCUM_STEPS     = 4            # Effective batch = BATCH_SIZE * GRAD_ACCUM = 8
LEARNING_RATE        = 2e-4         # Standard for LoRA. Reduce to 1e-4 if unstable
LR_SCHEDULER         = "cosine"     # Cosine annealing (recommended)
WARMUP_RATIO         = 0.03         # 3% of steps for warmup
WEIGHT_DECAY         = 0.001        # L2 regularization
MAX_SEQ_LENGTH       = 512          # Max tokens per sample (reduce for OOM)
OPTIMIZER            = "paged_adamw_8bit"  # Memory-efficient optimizer for QLoRA
SAVE_STEPS           = 25           # Save checkpoint every N steps
LOGGING_STEPS        = 5            # Log metrics every N steps
GRAD_CHECKPOINTING   = True         # Trade compute for memory (~30% VRAM savings)

# ── Generation Config ────────────────────────────────────
MAX_NEW_TOKENS   = 256              # Max tokens to generate at inference
TEMPERATURE      = 0.7              # Sampling temperature (0=greedy, 1=creative)
TOP_P            = 0.9              # Nucleus sampling

print("✅ Configuration set!")
print(f"   Base Model    : {BASE_MODEL}")
print(f"   LoRA Rank     : {LORA_R}")
print(f"   LoRA Alpha    : {LORA_ALPHA}  (scale = {LORA_ALPHA/LORA_R})")
print(f"   Effective LR  : {LEARNING_RATE}")
print(f"   Effective Batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   4-bit QLoRA   : {LOAD_IN_4BIT}")

✅ Configuration set!
   Base Model    : TinyLlama/TinyLlama-1.1B-Chat-v1.0
   LoRA Rank     : 16
   LoRA Alpha    : 32  (scale = 2.0)
   Effective LR  : 0.0002
   Effective Batch: 8
   4-bit QLoRA   : True


## Step 3: Load & Prepare Dataset

In [6]:
# ── Load the dataset ──────────────────────────────────────
print(f"Loading dataset: {DATASET_NAME}")
raw_dataset = load_dataset(DATASET_NAME, split="train")

print(f"Dataset size    : {len(raw_dataset)} samples")
print(f"Dataset columns : {raw_dataset.column_names}")
print("\nSample entry:")
print(raw_dataset[0])

Loading dataset: medalpaca/medical_meadow_wikidoc_patient_information


README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_patient_info.json:   0%|          | 0.00/3.49M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

Dataset size    : 5942 samples
Dataset columns : ['input', 'output', 'instruction']

Sample entry:
{'input': 'What are the symptoms of Allergy?', 'output': 'Allergy symptoms vary, but may include:\nBreathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing\nWhat part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:\nAllergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may cause itchy, watery, red, swollen eyes. Eating something you are allergic to can cause nausea, vomiting, abdominal pain, cramping, diarrhea, or a severe, life-threatening reaction. Allergens that touch the skin can cause a skin rash, hives, itching, blisters, or even skin p

In [7]:
# ── Format into instruction template ─────────────────────
# We use the Alpaca-style prompt format:
# ### Instruction:\n{input}\n\n### Response:\n{output}

def format_prompt(sample):
    """
    Format each sample into the instruction-following template.
    The model learns to produce 'output' given 'input' as context.

    KEY INSIGHT: The loss is computed on the full sequence (prompt + response),
    but ideally only on the response part (response-only masking).
    SFTTrainer with dataset_text_field handles this automatically.
    """
    instruction = sample.get('input', sample.get('question', ''))
    response    = sample.get('output', sample.get('answer', ''))

    return {
        "text": (
            f"### Instruction:\n{instruction}\n\n"
            f"### Response:\n{response}"
        ),
        "instruction": instruction,
        "response": response
    }

# Apply formatting
formatted_dataset = raw_dataset.map(format_prompt, remove_columns=raw_dataset.column_names)

# Limit samples for this demo
if MAX_TRAIN_SAMPLES:
    total = MAX_TRAIN_SAMPLES + MAX_VAL_SAMPLES
    formatted_dataset = formatted_dataset.select(range(min(total, len(formatted_dataset))))

# Train / validation split
split = formatted_dataset.train_test_split(test_size=TEST_SIZE, seed=42)
train_dataset = split["train"]
val_dataset   = split["test"]

print(f"\n✅ Formatted dataset ready:")
print(f"   Train samples : {len(train_dataset)}")
print(f"   Val samples   : {len(val_dataset)}")
print(f"\nSample formatted prompt:")
print("-" * 60)
print(train_dataset[0]["text"][:300], "...")

Map:   0%|          | 0/5942 [00:00<?, ? examples/s]


✅ Formatted dataset ready:
   Train samples : 540
   Val samples   : 60

Sample formatted prompt:
------------------------------------------------------------
### Instruction:
Who is at highest risk for Chronic stable angina ?

### Response:
Certain risk factors make it more likely that you will develop coronary artery disease (CAD) and subsequently present with anginal pain.
Major risk factors for stable angina that you can control include:
Smoking High  ...


## Step 4: Load Tokenizer

In [8]:
# ── Load Tokenizer ────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True  # Required for some models
)

# CRITICAL: Set padding side to 'right' for causal LMs
# Left padding would confuse the model during training
tokenizer.padding_side = "right"

# Add pad token if missing (LLaMA-based models often lack this)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("ℹ️  pad_token set to eos_token")

print(f"✅ Tokenizer loaded!")
print(f"   Vocab size    : {tokenizer.vocab_size:,}")
print(f"   Padding side  : {tokenizer.padding_side}")
print(f"   EOS token     : {tokenizer.eos_token!r} (id={tokenizer.eos_token_id})")
print(f"   BOS token     : {tokenizer.bos_token!r} (id={tokenizer.bos_token_id})")

# Quick test
test_tokens = tokenizer("Hello, how are you?", return_tensors="pt")
print(f"\nTest tokenization: {test_tokens.input_ids.shape} → {tokenizer.decode(test_tokens.input_ids[0])}")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

✅ Tokenizer loaded!
   Vocab size    : 32,000
   Padding side  : right
   EOS token     : '</s>' (id=2)
   BOS token     : '<s>' (id=1)

Test tokenization: torch.Size([1, 7]) → <s> Hello, how are you?


## Step 5: Load Quantized Base Model (QLoRA Setup)

> 💡 **What happens here**: The model weights are loaded in 4-bit NF4 format.  
> The original 7B model (28GB in fp32) becomes ~4GB in 4-bit. Magic!

In [9]:
# ── Step 5A: BitsAndBytes Quantization Config ─────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit              = LOAD_IN_4BIT,
    bnb_4bit_compute_dtype    = BNB_COMPUTE_DTYPE,
    bnb_4bit_quant_type       = BNB_QUANT_TYPE,
    bnb_4bit_use_double_quant = BNB_DOUBLE_QUANT,
)

print("✅ BitsAndBytesConfig:")
print(f"   4-bit loading       : {bnb_config.load_in_4bit}")
print(f"   Compute dtype       : {bnb_config.bnb_4bit_compute_dtype}")
print(f"   Quantization type   : {bnb_config.bnb_4bit_quant_type}")
print(f"   Double quantization : {bnb_config.bnb_4bit_use_double_quant}")

✅ BitsAndBytesConfig:
   4-bit loading       : True
   Compute dtype       : torch.float16
   Quantization type   : nf4
   Double quantization : True


In [10]:
# ── Step 5B: Load Base Model ──────────────────────────────
print(f"Loading {BASE_MODEL}...")
print("(This downloads the model and quantizes it — may take 2-5 min)")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",       # Automatically places layers on GPU/CPU
    trust_remote_code   = True,
    torch_dtype         = BNB_COMPUTE_DTYPE,
)

# CRITICAL for QLoRA: prepare model for k-bit training
# This unfreezes LayerNorm layers and casts them to fp32 for stability
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=GRAD_CHECKPOINTING
)

print(f"\n✅ Base model loaded!")

# Count total parameters
total_params = sum(p.numel() for p in base_model.parameters())
trainable_params_before = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params_before:,} (before LoRA: all frozen)")

# GPU memory usage
if torch.cuda.is_available():
    used_mem = torch.cuda.memory_allocated() / 1e9
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"   VRAM used        : {used_mem:.2f} GB / {total_mem:.1f} GB")

Loading TinyLlama/TinyLlama-1.1B-Chat-v1.0...
(This downloads the model and quantizes it — may take 2-5 min)


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


✅ Base model loaded!
   Total params     : 615,606,272
   Trainable params : 0 (before LoRA: all frozen)
   VRAM used        : 1.04 GB / 15.6 GB


## Step 6: Apply LoRA Adapters

> 🔑 This is the core PEFT step — injecting trainable adapter matrices into frozen base model layers.

In [11]:
# ── Step 6A: Define LoRA Configuration ───────────────────
lora_config = LoraConfig(
    r              = LORA_R,          # Rank of decomposition matrices
    lora_alpha     = LORA_ALPHA,      # Scaling: output *= alpha/r
    target_modules = TARGET_MODULES,  # Which layers get adapters
    lora_dropout   = LORA_DROPOUT,    # Dropout for regularization
    bias           = "none",          # Don't train bias terms
    task_type      = "CAUSAL_LM",     # Causal language modeling
)

print("✅ LoraConfig defined!")
print(f"   Rank (r)         : {lora_config.r}")
print(f"   Alpha            : {lora_config.lora_alpha}")
print(f"   Effective scale  : {lora_config.lora_alpha / lora_config.r:.2f}")
print(f"   Target modules   : {lora_config.target_modules}")
print(f"   Dropout          : {lora_config.lora_dropout}")

✅ LoraConfig defined!
   Rank (r)         : 16
   Alpha            : 32
   Effective scale  : 2.00
   Target modules   : {'o_proj', 'k_proj', 'v_proj', 'q_proj'}
   Dropout          : 0.05


In [17]:
# ── Step 6B: Apply LoRA to Model ─────────────────────────
model = get_peft_model(base_model, lora_config)

# Inspect what changed
total_params   = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params  = total_params - trainable_params

print("✅ LoRA adapters injected!")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,} ({100 * trainable_params / total_params:.3f}%)")
print(f"   Frozen params    : {frozen_params:,} ({100 * frozen_params / total_params:.3f}%)")

# Show model structure (LoRA layers)
print("\n📋 Trainable LoRA parameters by layer:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(f"   {name}: {param.shape} → {param.numel():,} params")

✅ LoRA adapters injected!
   Total params     : 620,111,872
   Trainable params : 4,505,600 (0.727%)
   Frozen params    : 615,606,272 (99.273%)

📋 Trainable LoRA parameters by layer:
   base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 2048]) → 32,768 params
   base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([2048, 16]) → 32,768 params
   base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 2048]) → 32,768 params
   base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16]) → 4,096 params
   base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([16, 2048]) → 32,768 params
   base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight: torch.Size([256, 16]) → 4,096 params
   base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight: torch.Size([16, 2048]) → 32,768 params
   base_model.model.mo

## Step 7: Define Training Arguments & Train

> 📊 Monitor the loss! It should decrease smoothly from ~2.5 → ~1.0–1.5 for a good fine-tuning run.

In [28]:
# ── Step 7A: Training Arguments ───────────────────────────
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = LR_SCHEDULER,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    optim                       = OPTIMIZER,
    bf16                        = True,           # Use fp16 for T4 (not bf16)
    gradient_checkpointing      = GRAD_CHECKPOINTING,
    logging_dir                 = f"{OUTPUT_DIR}/logs",
    logging_steps               = LOGGING_STEPS,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 3,              # Keep only last 3 checkpoints
    eval_strategy               = "steps",
    eval_steps                  = SAVE_STEPS,
    load_best_model_at_end      = True,
    report_to                   = "none",         # Disable wandb/tensorboard for simplicity
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
)

print("✅ TrainingArguments defined!")
eff_batch = BATCH_SIZE * GRAD_ACCUM_STEPS
steps_per_epoch = len(train_dataset) // eff_batch
total_steps = steps_per_epoch * NUM_EPOCHS
print(f"   Effective batch size     : {eff_batch}")
print(f"   Steps per epoch          : {steps_per_epoch}")
print(f"   Total training steps     : {total_steps}")
print(f"   Warmup steps             : {int(total_steps * WARMUP_RATIO)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


✅ TrainingArguments defined!
   Effective batch size     : 8
   Steps per epoch          : 67
   Total training steps     : 134
   Warmup steps             : 4


In [29]:
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset  = train_dataset,
    eval_dataset  = val_dataset,
    args=training_args
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [30]:
# ── Step 7C: START TRAINING 🚀 ────────────────────────────
print("🚀 Starting training...")
print("   Watch the training/eval loss in the output below.")
print("   A healthy run: loss decreasing smoothly each step.")
print("-" * 60)

# GPU memory before training
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    start_mem = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM before training: {start_mem:.2f} GB")

train_result = trainer.train()

# GPU memory after training
if torch.cuda.is_available():
    peak_mem = torch.cuda.max_memory_allocated() / 1e9
    print(f"\nPeak VRAM during training: {peak_mem:.2f} GB")

# Training stats
print("\n📊 Training Results:")
print(f"   Train Loss     : {train_result.training_loss:.4f}")
print(f"   Train Runtime  : {train_result.metrics['train_runtime']:.1f}s")
print(f"   Samples/sec    : {train_result.metrics['train_samples_per_second']:.2f}")
print(f"   Steps/sec      : {train_result.metrics['train_steps_per_second']:.3f}")

🚀 Starting training...
   Watch the training/eval loss in the output below.
   A healthy run: loss decreasing smoothly each step.
------------------------------------------------------------
VRAM before training: 1.08 GB


Step,Training Loss,Validation Loss
25,1.616305,1.656771
50,1.500653,1.571640
75,1.462208,1.547076
100,1.483766,1.537343
125,1.358244,1.534038



Peak VRAM during training: 3.78 GB

📊 Training Results:
   Train Loss     : 1.5483
   Train Runtime  : 309.6s
   Samples/sec    : 3.49
   Steps/sec      : 0.439


In [31]:
# ── Step 7D: Save the LoRA adapter ───────────────────────
# This saves ONLY the adapter weights (~50-200MB, NOT the full model)
print(f"Saving LoRA adapter to: {ADAPTER_DIR}")
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
adapter_files = os.listdir(ADAPTER_DIR)
print(f"\n✅ Saved files:")
for f in adapter_files:
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, f)) / 1e6
    print(f"   {f}: {size_mb:.1f} MB")

Saving LoRA adapter to: ./qlora-adapter-only

✅ Saved files:
   tokenizer_config.json: 0.0 MB
   adapter_model.safetensors: 18.0 MB
   README.md: 0.0 MB
   chat_template.jinja: 0.0 MB
   adapter_config.json: 0.0 MB
   tokenizer.json: 3.6 MB


## Step 8: Evaluation — Quantitative Metrics

> We compare BASE model vs FINE-TUNED model across 4 metrics:
> - **Perplexity**: How well the model predicts unseen text (lower = better)
> - **ROUGE-1/L**: N-gram overlap with reference responses (higher = better)
> - **BLEU**: Precision-based overlap (higher = better)
> - **Qualitative**: Side-by-side generation comparison

In [32]:
# ── Evaluation Helper Functions ───────────────────────────

def compute_perplexity(model, tokenizer, texts, max_length=256, batch_size=4):
    """
    Compute perplexity = exp(average cross-entropy loss on text).
    Lower perplexity = model finds text less 'surprising' = better fit.
    """
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        encodings = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=max_length,
            padding=True
        ).to(model.device)

        input_ids      = encodings.input_ids
        attention_mask = encodings.attention_mask

        with torch.no_grad():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=input_ids  # labels = input_ids for causal LM
            )

        # outputs.loss is mean CE loss across all tokens
        num_tokens = attention_mask.sum().item()
        total_loss   += outputs.loss.item() * num_tokens
        total_tokens += num_tokens

    avg_loss    = total_loss / total_tokens
    perplexity  = np.exp(avg_loss)
    return perplexity, avg_loss


def generate_response(model, tokenizer, instruction, max_new_tokens=200):
    """
    Generate model response for a given instruction.
    """
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens    = max_new_tokens,
            temperature       = TEMPERATURE,
            top_p             = TOP_P,
            do_sample         = True,
            pad_token_id      = tokenizer.eos_token_id,
            eos_token_id      = tokenizer.eos_token_id,
            repetition_penalty = 1.1,
        )

    # Decode only the generated part (not the prompt)
    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True)
    return response.strip()


def compute_rouge_bleu(predictions, references):
    """
    Compute ROUGE (1, 2, L) and BLEU scores.
    predictions: list of generated strings
    references : list of reference strings
    """
    rouge = evaluate.load("rouge")
    bleu  = evaluate.load("bleu")

    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_stemmer=True
    )

    # ✅ FIXED: pass raw strings, not pre-tokenized lists
    # evaluate's BLEU handles tokenization internally
    bleu_scores = bleu.compute(
        predictions=predictions,
        references=[[ref] for ref in references]  # each ref wrapped in a list
    )

    return rouge_scores, bleu_scores


print("✅ Evaluation functions defined!")

✅ Evaluation functions defined!


In [33]:
# ── Load the ORIGINAL base model for comparison ───────────
# We need the untuned model to compare against fine-tuned
print("Loading original base model for comparison...")
print("(Loading a SEPARATE untuned copy)")

base_model_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config = bnb_config,
    device_map          = "auto",
    trust_remote_code   = True,
    torch_dtype         = BNB_COMPUTE_DTYPE,
)

print("✅ Base model loaded for evaluation!")

Loading original base model for comparison...
(Loading a SEPARATE untuned copy)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✅ Base model loaded for evaluation!


In [34]:
# ── Metric 1: Perplexity Comparison ───────────────────────
print("📊 Computing Perplexity...")
print("(Lower = better — model finds eval text less 'surprising')")
print("-" * 50)

# Use validation set responses for perplexity
eval_texts = val_dataset["text"][:30]  # Use 30 samples for speed

# Fine-tuned model perplexity
ft_model = trainer.model
ppl_ft, loss_ft = compute_perplexity(ft_model, tokenizer, eval_texts)

# Base model perplexity
ppl_base, loss_base = compute_perplexity(base_model_eval, tokenizer, eval_texts)

ppl_improvement = ((ppl_base - ppl_ft) / ppl_base) * 100

print(f"\n   Base Model  Perplexity : {ppl_base:.2f}")
print(f"   Fine-tuned  Perplexity : {ppl_ft:.2f}")
print(f"   Improvement            : {ppl_improvement:.1f}% {'✅ Better' if ppl_improvement > 0 else '❌ Worse'}")

📊 Computing Perplexity...
(Lower = better — model finds eval text less 'surprising')
--------------------------------------------------

   Base Model  Perplexity : 2658.78
   Fine-tuned  Perplexity : 586.04
   Improvement            : 78.0% ✅ Better


In [37]:
# ── Metric 2: ROUGE & BLEU Scores ─────────────────────────
print("📊 Computing ROUGE and BLEU...")
print("(Using 20 eval samples for speed — increase for real eval)")
print("-" * 50)

N_EVAL = 20  # Increase to 100 for more reliable scores
eval_samples = val_dataset.select(range(N_EVAL))

base_predictions = []
ft_predictions   = []
references       = []

for i, sample in enumerate(eval_samples):
    instruction = sample["instruction"]
    reference   = sample["response"]

    # Generate from both models
    base_pred = generate_response(base_model_eval, tokenizer, instruction, max_new_tokens=150)
    ft_pred   = generate_response(ft_model, tokenizer, instruction, max_new_tokens=150)

    base_predictions.append(base_pred)
    ft_predictions.append(ft_pred)
    references.append(reference)

    if (i + 1) % 5 == 0:
        print(f"   Processed {i+1}/{N_EVAL} samples...")

# Compute scores
rouge_base, bleu_base = compute_rouge_bleu(base_predictions, references)
rouge_ft,   bleu_ft   = compute_rouge_bleu(ft_predictions,   references)

print("\n✅ Generation complete!")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📊 Computing ROUGE and BLEU...
(Using 20 eval samples for speed — increase for real eval)
--------------------------------------------------


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   Processed 5/20 samples...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   Processed 10/20 samples...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   Processed 15/20 samples...


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

   Processed 20/20 samples...

✅ Generation complete!


In [38]:
# ── Full Metrics Summary Table ─────────────────────────────
print("\n" + "═" * 65)
print("  📊  EVALUATION RESULTS: BASE MODEL vs FINE-TUNED MODEL")
print("═" * 65)
print(f"{'Metric':<22} {'Base Model':>14} {'Fine-Tuned':>14} {'Change':>10}")
print("-" * 65)

# Perplexity
ppl_change = ppl_base - ppl_ft
print(f"{'Perplexity (↓)':<22} {ppl_base:>14.2f} {ppl_ft:>14.2f} {ppl_change:>+10.2f}")

# ROUGE scores
for key in ["rouge1", "rouge2", "rougeL"]:
    b = rouge_base[key]
    f = rouge_ft[key]
    delta = f - b
    label = f"{key.upper()} (↑)"
    print(f"{label:<22} {b:>14.4f} {f:>14.4f} {delta:>+10.4f}")

# BLEU
b_bleu = bleu_base["bleu"]
f_bleu = bleu_ft["bleu"]
delta_bleu = f_bleu - b_bleu
print(f"{'BLEU (↑)':<22} {b_bleu:>14.4f} {f_bleu:>14.4f} {delta_bleu:>+10.4f}")

print("═" * 65)
print("  ↓ = lower is better  |  ↑ = higher is better")
print("  Positive Perplexity change = fine-tuned is better")
print("  Positive ROUGE/BLEU change = fine-tuned is better")
print("═" * 65)

# Overall verdict
improvements = sum([
    ppl_ft < ppl_base,
    rouge_ft["rouge1"] > rouge_base["rouge1"],
    rouge_ft["rougeL"] > rouge_base["rougeL"],
    f_bleu > b_bleu,
])

print(f"\n🏁 Verdict: Fine-tuned model improves {improvements}/4 metrics.")
if improvements >= 3:
    print("   ✅ Fine-tuning was SUCCESSFUL!")
elif improvements == 2:
    print("   ⚠️  Mixed results. Consider more data or hyperparameter tuning.")
else:
    print("   ❌ Fine-tuning may have underfit or overfit. Review data & params.")


═════════════════════════════════════════════════════════════════
  📊  EVALUATION RESULTS: BASE MODEL vs FINE-TUNED MODEL
═════════════════════════════════════════════════════════════════
Metric                     Base Model     Fine-Tuned     Change
-----------------------------------------------------------------
Perplexity (↓)                2658.78         586.04   +2072.74
ROUGE1 (↑)                     0.2515         0.2634    +0.0119
ROUGE2 (↑)                     0.0509         0.0612    +0.0103
ROUGEL (↑)                     0.1395         0.1614    +0.0219
BLEU (↑)                       0.0219         0.0252    +0.0033
═════════════════════════════════════════════════════════════════
  ↓ = lower is better  |  ↑ = higher is better
  Positive Perplexity change = fine-tuned is better
  Positive ROUGE/BLEU change = fine-tuned is better
═════════════════════════════════════════════════════════════════

🏁 Verdict: Fine-tuned model improves 4/4 metrics.
   ✅ Fine-tuning was SUCCES

## Step 9: Qualitative Evaluation — Side-by-Side Comparison

In [39]:
# ── Qualitative Comparison: 5 test questions ──────────────
# Try your own domain-specific questions here!
test_questions = [
    "What are the common symptoms of Type 2 Diabetes?",
    "How does aspirin help prevent heart attacks?",
    "What is the difference between a virus and a bacteria?",
    "What should I do if I have a high fever?",
    "How does hypertension affect the kidneys?",
]

print("=" * 70)
print("  🔬  QUALITATIVE COMPARISON: Base vs Fine-Tuned")
print("=" * 70)

for i, question in enumerate(test_questions):
    print(f"\n{'─' * 70}")
    print(f"Q{i+1}: {question}")
    print(f"{'─' * 70}")

    base_ans = generate_response(base_model_eval, tokenizer, question, max_new_tokens=150)
    ft_ans   = generate_response(ft_model, tokenizer, question, max_new_tokens=150)

    print(f"\n🔵 BASE MODEL:")
    print(f"   {base_ans[:300]}..." if len(base_ans) > 300 else f"   {base_ans}")

    print(f"\n🟢 FINE-TUNED:")
    print(f"   {ft_ans[:300]}..." if len(ft_ans) > 300 else f"   {ft_ans}")

print(f"\n{'=' * 70}")

Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  🔬  QUALITATIVE COMPARISON: Base vs Fine-Tuned

──────────────────────────────────────────────────────────────────────
Q1: What are the common symptoms of Type 2 Diabetes?
──────────────────────────────────────────────────────────────────────


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔵 BASE MODEL:
   Yes, here are some common symptoms of type 2 diabetes:
1. High blood sugar levels (hyperglycemia)
2. Blurred vision or difficulty seeing clearly
3. Sore, tender, or swollen feet, hands, or legs
4. Muscle weakness and fatigue
5. Dehydration
6. Dark urine
7. Poor sleep quality
8. Nausea, vomiting, or ...

🟢 FINE-TUNED:
   Symptoms may appear as little or no symptom, or they may be severe. Common symptoms of type 1 and type 2 diabetes include:
Diabetic ketoacidosis (DKA) Fatigue Hypoglycemia (low blood sugar) Increased thirst and urination Numbness or tingling of the hands, feet, legs, arms, or face (peripheral neurop...

──────────────────────────────────────────────────────────────────────
Q2: How does aspirin help prevent heart attacks?
──────────────────────────────────────────────────────────────────────


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


🔵 BASE MODEL:
   A virus is a self-replicating organism that infects and replicates within host cells. It does not have a nucleus, but rather a non-coding RNA genome that directs the cell to produce proteins. A bacteria on the other hand has a centralized, double-layered DNA genome that contains both the genetic mat...

🟢 FINE-TUNED:
   Both viruses and bacteria are organisms that reproduce by the process of infection. However, there are differences in how these organisms reproduce.

──────────────────────────────────────────────────────────────────────
Q4: What should I do if I have a high fever?
──────────────────────────────────────────────────────────────────────


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔵 BASE MODEL:
   If you have a high fever, it's best to contact your doctor as soon as possible. High fevers can indicate an infection or other health problem that requires immediate attention. Your doctor will want to know the cause of your fever, and will likely conduct a physical examination, blood tests, and X-r...

🟢 FINE-TUNED:
   If your child is feeling very hot and then cold, it may be caused by an infection. Your child's fever may be caused by:
Colds from the nose or throat Influenza (the flu) Rheumatic fever
Your child should be examined for any other causes of high fevers that are not due to an infection.

──────────────────────────────────────────────────────────────────────
Q5: How does hypertension affect the kidneys?
──────────────────────────────────────────────────────────────────────


Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔵 BASE MODEL:
   Hypertension can affect the kidneys by causing damage to the blood vessels, which supply blood to the kidneys. This damage leads to a reduction in blood flow and impaired kidney function. High blood pressure also increases the flow of waste products into the kidneys, leading to further kidney damage...

🟢 FINE-TUNED:
   Hypertension can damage the kidney tissue. The kidneys are small and may be damaged if blood pressure is too high. High blood pressure causes the blood vessels to narrow, making it harder for blood to flow to the kidneys. This makes it difficult for the kidneys to filter waste products from the bloo...



## Step 10: Loss Curve Visualization

In [ ]:
# ── Plot Training Loss Curve ──────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0b0f1a'
matplotlib.rcParams['axes.facecolor']   = '#111827'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#64748b'
matplotlib.rcParams['ytick.color']      = '#64748b'

# Extract training history
history = trainer.state.log_history

train_steps = [h['step'] for h in history if 'loss' in h]
train_losses = [h['loss'] for h in history if 'loss' in h]
eval_steps  = [h['step'] for h in history if 'eval_loss' in h]
eval_losses = [h['eval_loss'] for h in history if 'eval_loss' in h]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training Diagnostics — QLoRA Fine-Tuning', fontsize=14, color='white', y=1.02)

# Loss curves
ax1 = axes[0]
ax1.plot(train_steps, train_losses, color='#38bdf8', linewidth=2, label='Train Loss')
if eval_losses:
    ax1.plot(eval_steps, eval_losses, color='#f472b6', linewidth=2, linestyle='--', label='Eval Loss')
ax1.set_xlabel('Step', color='#94a3b8')
ax1.set_ylabel('Loss', color='#94a3b8')
ax1.set_title('Training & Validation Loss', color='white')
ax1.legend(facecolor='#1a2235', edgecolor='#1e2d45')
ax1.spines['bottom'].set_color('#1e2d45')
ax1.spines['left'].set_color('#1e2d45')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Metrics bar chart
ax2 = axes[1]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L', 'BLEU']
base_vals = [
    rouge_base['rouge1'], rouge_base['rouge2'],
    rouge_base['rougeL'], b_bleu
]
ft_vals = [
    rouge_ft['rouge1'], rouge_ft['rouge2'],
    rouge_ft['rougeL'], f_bleu
]

x = np.arange(len(metrics))
w = 0.35
bars1 = ax2.bar(x - w/2, base_vals, w, label='Base Model', color='#475569', alpha=0.8)
bars2 = ax2.bar(x + w/2, ft_vals,   w, label='Fine-Tuned',  color='#38bdf8', alpha=0.9)

ax2.set_xticks(x)
ax2.set_xticklabels(metrics)
ax2.set_ylabel('Score', color='#94a3b8')
ax2.set_title('ROUGE & BLEU Comparison', color='white')
ax2.legend(facecolor='#1a2235', edgecolor='#1e2d45')
ax2.spines['bottom'].set_color('#1e2d45')
ax2.spines['left'].set_color('#1e2d45')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('training_results.png', dpi=150, bbox_inches='tight',
            facecolor='#0b0f1a', edgecolor='none')
plt.show()
print("✅ Plot saved as training_results.png")

## Step 11: Merge Adapter into Base Model (Optional)

> This creates a single merged model file. Use for deployment.  
> ⚠️ Requires enough RAM/VRAM to load both base + adapter + merged model simultaneously.

In [ ]:
# ── Optional: Merge LoRA adapter back into base model ──────
# After merging, you get a standard HuggingFace model with no adapter dependency.
# Useful for: vLLM deployment, GGUF conversion, sharing on HuggingFace Hub.

MERGE_AND_SAVE = False  # Set to True to merge (needs more RAM)
MERGED_DIR = "./merged-model"

if MERGE_AND_SAVE:
    print("Merging LoRA adapter into base model...")

    # Load base in fp16 for merging (not quantized)
    merge_base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype       = torch.float16,
        device_map        = "auto",
        trust_remote_code = True,
    )

    # Load adapter
    merged = PeftModel.from_pretrained(merge_base, ADAPTER_DIR)

    # Merge weights — this permanently fuses A and B into W
    merged = merged.merge_and_unload()

    # Save merged model
    merged.save_pretrained(MERGED_DIR)
    tokenizer.save_pretrained(MERGED_DIR)

    print(f"✅ Merged model saved to {MERGED_DIR}")
    print(f"   This model can be loaded with standard AutoModelForCausalLM")
    print(f"   No PEFT library needed at inference time!")
else:
    print("ℹ️  Merge skipped. Set MERGE_AND_SAVE=True to enable.")
    print("   To load adapter at inference time:")
    print("   model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, ...)")
    print("   model = PeftModel.from_pretrained(model, ADAPTER_DIR)")

## Step 12: Inference — Using Your Fine-Tuned Model

In [40]:
# ── Interactive Inference ─────────────────────────────────
# Try your own questions here!

def ask_finetuned(question, verbose=True):
    """Simple wrapper to query the fine-tuned model."""
    answer = generate_response(trainer.model, tokenizer, question, max_new_tokens=200)
    if verbose:
        print(f"🟡 Question : {question}")
        print(f"🟢 Answer   : {answer}")
        print()
    return answer

# ── Try these examples ──
ask_finetuned("What are the side effects of ibuprofen?")
ask_finetuned("Explain what cholesterol does in the body.")
ask_finetuned("What is the difference between Type 1 and Type 2 diabetes?")

Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟡 Question : What are the side effects of ibuprofen?
🟢 Answer   : There is no specific information available on the potential side effects of ibuprofen. However, it is always better to be cautious when taking any medication, especially if you have health conditions such as kidney disease, liver problems, high blood pressure or other conditions that can cause side effects.



Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🟡 Question : Explain what cholesterol does in the body.
🟢 Answer   : Cholesterol is a fatty substance found in your blood and body tissues. It helps your body absorb nutrients, particularly those from foods high in fat or protein. In some people, high levels of cholesterol can lead to heart disease, artery blockages, and other health problems.
When cholesterol is not needed, it leaves the body through the liver and kidneys. This process is called "metabolism."

🟡 Question : What is the difference between Type 1 and Type 2 diabetes?
🟢 Answer   : Type 1 diabetes is a condition in which the immune system attacks and destroys the pancreas. The body's cells begin producing insulin, but the immune system doesn't recognize it as an "invader" and tries to destroy it. It causes high blood sugar levels. In type 2 diabetes, the immune system either doesn't produce enough insulin or can't use the insulin produced by the pancreas. Both types of diabetes are metabolic disorders that affect the way y

'Type 1 diabetes is a condition in which the immune system attacks and destroys the pancreas. The body\'s cells begin producing insulin, but the immune system doesn\'t recognize it as an "invader" and tries to destroy it. It causes high blood sugar levels. In type 2 diabetes, the immune system either doesn\'t produce enough insulin or can\'t use the insulin produced by the pancreas. Both types of diabetes are metabolic disorders that affect the way your body uses food and carbohydrates for energy.'

---
## 📚 Summary & Key Takeaways

```
┌──────────────────────────────────────────────────────────────┐
│                  WHAT WE DID IN THIS NOTEBOOK                │
├──────────────────────────────────────────────────────────────┤
│  1. Loaded TinyLlama-1.1B in 4-bit (QLoRA)                  │
│  2. Applied LoRA adapters (rank=16) to attention layers      │
│  3. Trained ONLY the adapter weights (~0.3% of params)       │
│  4. Evaluated with: Perplexity, ROUGE-1/2/L, BLEU           │
│  5. Side-by-side qualitative comparison                      │
│  6. Saved lightweight adapter (~50MB vs 2.2GB full model)    │
└──────────────────────────────────────────────────────────────┘

KEY HYPERPARAMETERS TO TUNE:
  LoRA rank (r)    → 8 (fast) to 64 (powerful)
  lora_alpha       → 2× rank for stability
  learning_rate    → 1e-4 to 5e-4 for LoRA
  max_seq_length   → reduce if OOM (512 → 256)
  num_epochs       → 1-3 for instruction tuning
  target_modules   → add gate/up/down_proj for more capacity

TO SCALE UP:
  - Use Mistral-7B-Instruct or Llama-3.1-8B with T4 + QLoRA
  - Use Llama-3.1-70B with 2× A100 80GB + QLoRA
  - Add more training data (1K → 10K samples for better results)
  - Enable response-only masking (DataCollatorForCompletionOnlyLM)
```

**References:**
- LoRA Paper: Hu et al. 2021 — https://arxiv.org/abs/2106.09685
- QLoRA Paper: Dettmers et al. 2023 — https://arxiv.org/abs/2305.14314
- Hugging Face PEFT: https://github.com/huggingface/peft
- Codebasics Course: https://github.com/codebasics/llm-fine-tuning-crash-course